In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-23 16:47:44,590]::3827333478::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [6]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
train_dataloader = DataLoader(dataset=train_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
val_dataloader = DataLoader(dataset=val_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
max_value=0
print(1)

1


In [11]:
# Multi-seed training initializes a fresh model and optimizer inside the training cell.
# MSE is used as both the training objective and validation model-selection metric.
criterion = torch.nn.MSELoss().to(device)
print("Criterion: MSELoss")


Criterion: MSELoss


In [12]:
# Optional model summary (uncomment if needed).
# from torchinfo import summary
# summary(SwinTransformer3D().to(device),
#         input_size=(4, 3, 15, 224, 224),
#         col_names=['input_size', 'output_size', 'num_params'])


In [13]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [14]:
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

all_seed_results = []

# Make sure the checkpoint directory exists.
os.makedirs('./save_swintransformer_folder', exist_ok=True)

with torch.cuda.device(0):
    for seed in SEEDS:
        print("\n" + "=" * 80)
        print(f"START SEED {seed} | Patience = {patience}")
        print("=" * 80)

        set_seed(seed)

        # A fresh shuffled training loader for each seed.
        generator = torch.Generator()
        generator.manual_seed(seed)

        train_dataloader_seed = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # Validation does not need shuffling.
        val_dataloader_seed = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker
        )

        # IMPORTANT: independently initialize model and optimizer for every seed.
        model = SwinTransformer3D().to(device)
        criterion = torch.nn.MSELoss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

        best_mse = float('inf')
        best_epoch = 0
        no_improve_count = 0

        # Exactly one best checkpoint is retained for each seed.
        best_model_path = f'./save_swintransformer_folder/best_model_seed_{seed}.pth'

        for i in range(epochs):
            # -------------------- Train --------------------
            model.train()
            train_avg_loss = 0.0

            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader_seed,
                desc=f"Seed {seed} Train {i + 1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                audiodata = audiodata.to(device)
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()
                hypothesis = model(fullshot)
                loss = criterion(hypothesis, big_five_data)
                loss.backward()
                optimizer.step()

                train_avg_loss += loss.item()

            train_avg_loss /= len(train_dataloader_seed)

            # -------------------- Validation --------------------
            model.eval()
            val_avg_loss = 0.0
            val_avg_o_loss = 0.0
            val_avg_c_loss = 0.0
            val_avg_e_loss = 0.0
            val_avg_a_loss = 0.0
            val_avg_n_loss = 0.0

            with torch.no_grad():
                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader_seed,
                    desc=f"Seed {seed} Val {i + 1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                    audiodata = audiodata.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(fullshot)
                    val_loss = criterion(hypothesis, big_five_data)

                    # Overall validation MSE.
                    val_avg_loss += val_loss.item()

                    # Per-trait validation MSE.
                    # Direct tensor slicing keeps these values on the same device
                    # and avoids rebuilding CPU tensors via extract_ocean().
                    val_avg_o_loss += criterion(hypothesis[..., 0], big_five_data[..., 0]).item()
                    val_avg_c_loss += criterion(hypothesis[..., 1], big_five_data[..., 1]).item()
                    val_avg_e_loss += criterion(hypothesis[..., 2], big_five_data[..., 2]).item()
                    val_avg_a_loss += criterion(hypothesis[..., 3], big_five_data[..., 3]).item()
                    val_avg_n_loss += criterion(hypothesis[..., 4], big_five_data[..., 4]).item()

            num_val_batches = len(val_dataloader_seed)
            val_avg_loss /= num_val_batches
            val_avg_o_loss /= num_val_batches
            val_avg_c_loss /= num_val_batches
            val_avg_e_loss /= num_val_batches
            val_avg_a_loss /= num_val_batches
            val_avg_n_loss /= num_val_batches

            # -------------------- Best MSE / Early stopping --------------------
            if val_avg_loss < best_mse:
                best_mse = val_avg_loss
                best_epoch = i + 1
                no_improve_count = 0

                torch.save({
                    'seed': seed,
                    'epoch': best_epoch,
                    'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'mse': best_mse,
                }, best_model_path)
            else:
                no_improve_count += 1

            # MSE only: no 1-MSE / 1-MAE conversion.
            logger.debug(
                'Seed: {} | Epoch: {} | Train MSE: {:.6f} | '
                'Val MSE: {:.6f} | Best MSE: {:.6f} | Patience: {}/{} | '
                'O MSE: {:.6f} | C MSE: {:.6f} | E MSE: {:.6f} | '
                'A MSE: {:.6f} | N MSE: {:.6f}'.format(
                    seed,
                    i + 1,
                    train_avg_loss,
                    val_avg_loss,
                    best_mse,
                    no_improve_count,
                    patience,
                    val_avg_o_loss,
                    val_avg_c_loss,
                    val_avg_e_loss,
                    val_avg_a_loss,
                    val_avg_n_loss
                )
            )

            print(
                f"Seed {seed} | Epoch {i + 1:03d} | "
                f"Train MSE={train_avg_loss:.6f} | "
                f"Val MSE={val_avg_loss:.6f} | "
                f"Best MSE={best_mse:.6f} | "
                f"Patience={no_improve_count}/{patience}"
            )

            if no_improve_count >= patience:
                logger.info(
                    'Seed {} early stopping at epoch {} | '
                    'Best Epoch: {} | Best MSE: {:.6f} | Patience: {}/{}'.format(
                        seed,
                        i + 1,
                        best_epoch,
                        best_mse,
                        no_improve_count,
                        patience
                    )
                )
                break

            torch.cuda.empty_cache()

        all_seed_results.append({
            'seed': seed,
            'best_epoch': best_epoch,
            'best_mse': best_mse,
            'checkpoint': best_model_path
        })

        print(
            f"\nSeed {seed} finished | "
            f"Best Epoch={best_epoch} | Best MSE={best_mse:.6f} | "
            f"Saved={best_model_path}"
        )

        del model
        del optimizer
        gc.collect()
        torch.cuda.empty_cache()

# -------------------- Final 5-seed summary --------------------
best_mses = np.array(
    [result['best_mse'] for result in all_seed_results],
    dtype=np.float64
)

print("\n" + "=" * 80)
print("FINAL 5-SEED MSE RESULTS")
print("=" * 80)

for result in all_seed_results:
    print(
        f"Seed {result['seed']:4d} | "
        f"Best Epoch={result['best_epoch']:3d} | "
        f"Best MSE={result['best_mse']:.6f}"
    )

print("-" * 80)
print(f"Mean MSE = {best_mses.mean():.6f}")
print(f"Std  MSE = {best_mses.std(ddof=1):.6f}")
print(f"MSE = {best_mses.mean():.6f} ± {best_mses.std(ddof=1):.6f}")



START SEED 42 | Patience = 20


Seed 42 Val 1/120: 100%|██████████| 191/191 [00:14<00:00, 12.74it/s]
[2026-09-23 16:52:21,123]::3675980870::DEBUG::Seed: 42 | Epoch: 1 | Train MSE: 0.943834 | Val MSE: 1.565860 | Best MSE: 1.565860 | Patience: 0/20 | O MSE: 1.353024 | C MSE: 1.276311 | E MSE: 1.630316 | A MSE: 1.639566 | N MSE: 1.930080


Seed 42 | Epoch 001 | Train MSE=0.943834 | Val MSE=1.565860 | Best MSE=1.565860 | Patience=0/20


Seed 42 Val 2/120: 100%|██████████| 191/191 [00:15<00:00, 12.63it/s]
[2026-09-23 16:56:52,463]::3675980870::DEBUG::Seed: 42 | Epoch: 2 | Train MSE: 0.207141 | Val MSE: 1.315835 | Best MSE: 1.315835 | Patience: 0/20 | O MSE: 1.149014 | C MSE: 1.417415 | E MSE: 1.264764 | A MSE: 1.486955 | N MSE: 1.261025


Seed 42 | Epoch 002 | Train MSE=0.207141 | Val MSE=1.315835 | Best MSE=1.315835 | Patience=0/20


...

Seed 2024 | Epoch 024 | Train MSE=0.003948 | Val MSE=1.411746 | Best MSE=1.330638 | Patience=18/20


Seed 2024 Val 25/120: 100%|██████████| 191/191 [00:15<00:00, 12.18it/s]
[2026-09-24 03:33:37,657]::3675980870::DEBUG::Seed: 2024 | Epoch: 25 | Train MSE: 0.004907 | Val MSE: 1.397657 | Best MSE: 1.330638 | Patience: 19/20 | O MSE: 1.021288 | C MSE: 1.402459 | E MSE: 1.402716 | A MSE: 1.638077 | N MSE: 1.523744


Seed 2024 | Epoch 025 | Train MSE=0.004907 | Val MSE=1.397657 | Best MSE=1.330638 | Patience=19/20


Seed 2024 Val 26/120: 100%|██████████| 191/191 [00:15<00:00, 12.02it/s]
[2026-09-24 03:38:12,863]::3675980870::DEBUG::Seed: 2024 | Epoch: 26 | Train MSE: 0.006378 | Val MSE: 1.434242 | Best MSE: 1.330638 | Patience: 20/20 | O MSE: 1.132667 | C MSE: 1.507480 | E MSE: 1.334223 | A MSE: 1.605768 | N MSE: 1.591075
[2026-09-24 03:38:12,864]::3675980870::INFO::Seed 2024 early stopping at epoch 26 | Best Epoch: 6 | Best MSE: 1.330638 | Patience: 20/20


Seed 2024 | Epoch 026 | Train MSE=0.006378 | Val MSE=1.434242 | Best MSE=1.330638 | Patience=20/20

Seed 2024 finished | Best Epoch=6 | Best MSE=1.330638 | Saved=./save_swintransformer_folder/best_model_seed_2024.pth

FINAL 5-SEED MSE RESULTS
Seed   42 | Best Epoch=  2 | Best MSE=1.315835
Seed  123 | Best Epoch=  7 | Best MSE=1.341857
Seed  777 | Best Epoch= 17 | Best MSE=1.315789
Seed 1004 | Best Epoch= 10 | Best MSE=1.349034
Seed 2024 | Best Epoch=  6 | Best MSE=1.330638
--------------------------------------------------------------------------------
Mean MSE = 1.330631
Std  MSE = 0.015033
MSE = 1.330631 ± 0.015033
